# Ensemble Methods: XGBoost vs Random Forest — Car Evaluation Dataset

---

## Copyright and License

This Jupyter Notebook is provided for educational and research purposes.

## Disclaimer

This notebook is provided "as is", without warranty of any kind, express or implied.  
All analyses and interpretations are for educational and research purposes only.

## Dataset Note

**Dataset:** Car Evaluation  
**Source:** https://www.kaggle.com/datasets/elikplim/car-evaluation-data-set  
**Locally under:** `../datasets/car_evaluation.csv`  
**License:** CC0: Public Domain

---

## Abstract

This notebook presents a comparative study of two leading ensemble methods — **Random Forest** (bagging) and **XGBoost** (gradient boosting) — applied to the Car Evaluation multi-class classification problem. Both models are tuned via `GridSearchCV` and `RandomizedSearchCV`, and compared across accuracy, ROC-AUC, training time, and learning curves. Visualisations include confusion matrices, feature importances, ROC curves (OvR), and hyperparameter sensitivity plots.

---

## Table of Contents

1. [Import Libraries](#1-import-libraries)  
2. [Load & Preprocess Data](#2-load--preprocess-data)  
3. [Baseline Models](#3-baseline-models)  
4. [Hyperparameter Tuning — Random Forest](#4-hyperparameter-tuning--random-forest)  
5. [Hyperparameter Tuning — XGBoost](#5-hyperparameter-tuning--xgboost)  
6. [Model Comparison](#6-model-comparison)  
7. [ROC Curves](#7-roc-curves)  
8. [Learning Curves](#8-learning-curves)  
9. [Feature Importance](#9-feature-importance)  
10. [Conclusions](#10-conclusions)

## 1. Import Libraries

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import randint, uniform

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, label_binarize
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    cross_val_score, learning_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
)
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 2. Load & Preprocess Data

In [ ]:
col_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv('../datasets/car_evaluation.csv', header=None, names=col_names)

features = [c for c in df.columns if c != 'class']

# Ordinal encoding with domain-aware order
enc = OrdinalEncoder(categories=[
    ['low', 'med', 'high', 'vhigh'],
    ['low', 'med', 'high', 'vhigh'],
    ['2', '3', '4', '5more'],
    ['2', '4', 'more'],
    ['small', 'med', 'big'],
    ['low', 'med', 'high']
])
X = pd.DataFrame(enc.fit_transform(df[features]), columns=features)

le = LabelEncoder()
y = le.fit_transform(df['class'])
n_classes = len(le.classes_)
print('Classes:', dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 3. Baseline Models

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name):
    """Fit, time, and evaluate a model. Returns a result dict."""
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)
    acc  = accuracy_score(y_te, y_pred)
    rauc = roc_auc_score(y_te, y_prob, multi_class='ovr', average='macro')
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='accuracy').mean()

    print(f'\n=== {name} ===')
    print(f'  Test Accuracy : {acc:.4f}')
    print(f'  CV Accuracy   : {cv:.4f}')
    print(f'  ROC-AUC (OvR) : {rauc:.4f}')
    print(f'  Train time    : {train_time:.2f}s')
    print(classification_report(y_te, y_pred, target_names=le.classes_))

    return {'name': name, 'model': model, 'acc': acc, 'cv': cv,
            'roc_auc': rauc, 'time': train_time, 'y_pred': y_pred, 'y_prob': y_prob}

rf_base  = RandomForestClassifier(random_state=SEED, n_jobs=-1)
xgb_base = XGBClassifier(random_state=SEED, eval_metric='mlogloss',
                          use_label_encoder=False, n_jobs=-1)

res_rf_base  = evaluate(rf_base,  X_train, y_train, X_test, y_test, 'Random Forest (baseline)')
res_xgb_base = evaluate(xgb_base, X_train, y_train, X_test, y_test, 'XGBoost (baseline)')

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, res in zip(axes, [res_rf_base, res_xgb_base]):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(res['name'])
plt.suptitle('Baseline Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning — Random Forest

In [ ]:
# GridSearchCV — Random Forest
rf_grid = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2']
}
rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    rf_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
rf_gs.fit(X_train, y_train)
print('Best RF params:', rf_gs.best_params_)
print(f'Best RF CV accuracy: {rf_gs.best_score_:.4f}')

In [ ]:
# Sensitivity: n_estimators vs CV accuracy (max_features=sqrt)
rf_results = pd.DataFrame(rf_gs.cv_results_)
rf_sub = rf_results[
    (rf_results['param_max_features'] == 'sqrt') &
    (rf_results['param_min_samples_split'] == 2)
].copy()

fig, ax = plt.subplots(figsize=(9, 5))
for depth in [5, 10, 20, None]:
    s = rf_sub[rf_sub['param_max_depth'] == depth]
    label = f'max_depth={depth}'
    ax.plot(s['param_n_estimators'].astype(str), s['mean_test_score'],
            marker='o', label=label)
ax.set_title('RF: n_estimators vs CV Accuracy (max_features=sqrt)')
ax.set_xlabel('n_estimators')
ax.set_ylabel('Mean CV Accuracy')
ax.legend(title='max_depth')
plt.tight_layout()
plt.show()

In [ ]:
# RandomizedSearchCV — Random Forest
rf_rand_dist = {
    'n_estimators':      randint(50, 400),
    'max_depth':         [5, 8, 10, 15, 20, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf':  randint(1, 10),
    'max_features':      ['sqrt', 'log2']
}
rf_rs = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    rf_rand_dist, n_iter=60, cv=5, scoring='accuracy',
    random_state=SEED, n_jobs=-1, verbose=1
)
rf_rs.fit(X_train, y_train)
print('Best RF (random) params:', rf_rs.best_params_)
print(f'Best RF (random) CV accuracy: {rf_rs.best_score_:.4f}')

# Pick best RF overall
best_rf = rf_gs.best_estimator_ if rf_gs.best_score_ >= rf_rs.best_score_ else rf_rs.best_estimator_
res_rf_tuned = evaluate(best_rf, X_train, y_train, X_test, y_test, 'Random Forest (tuned)')

## 5. Hyperparameter Tuning — XGBoost

In [ ]:
# GridSearchCV — XGBoost
xgb_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [3, 5, 7],
    'learning_rate':[0.05, 0.1, 0.3],
    'subsample':    [0.7, 1.0]
}
xgb_gs = GridSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='mlogloss',
                  use_label_encoder=False, n_jobs=-1),
    xgb_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
xgb_gs.fit(X_train, y_train)
print('Best XGB params:', xgb_gs.best_params_)
print(f'Best XGB CV accuracy: {xgb_gs.best_score_:.4f}')

In [ ]:
# Sensitivity: learning_rate vs CV accuracy per max_depth
xgb_results = pd.DataFrame(xgb_gs.cv_results_)
xgb_sub = xgb_results[
    (xgb_results['param_n_estimators'] == 100) &
    (xgb_results['param_subsample'] == 1.0)
].copy()

fig, ax = plt.subplots(figsize=(9, 5))
for depth in [3, 5, 7]:
    s = xgb_sub[xgb_sub['param_max_depth'] == depth]
    ax.plot(s['param_learning_rate'].astype(str), s['mean_test_score'],
            marker='o', label=f'max_depth={depth}')
ax.set_title('XGBoost: learning_rate vs CV Accuracy (n_estimators=100)')
ax.set_xlabel('learning_rate')
ax.set_ylabel('Mean CV Accuracy')
ax.legend(title='max_depth')
plt.tight_layout()
plt.show()

In [ ]:
# RandomizedSearchCV — XGBoost
xgb_rand_dist = {
    'n_estimators':  randint(50, 400),
    'max_depth':     randint(3, 10),
    'learning_rate': uniform(0.01, 0.4),
    'subsample':     uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma':         uniform(0, 0.5),
    'reg_alpha':     uniform(0, 1),
    'reg_lambda':    uniform(0.5, 2)
}
xgb_rs = RandomizedSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='mlogloss',
                  use_label_encoder=False, n_jobs=-1),
    xgb_rand_dist, n_iter=60, cv=5, scoring='accuracy',
    random_state=SEED, n_jobs=-1, verbose=1
)
xgb_rs.fit(X_train, y_train)
print('Best XGB (random) params:', xgb_rs.best_params_)
print(f'Best XGB (random) CV accuracy: {xgb_rs.best_score_:.4f}')

# Pick best XGB overall
best_xgb = xgb_gs.best_estimator_ if xgb_gs.best_score_ >= xgb_rs.best_score_ else xgb_rs.best_estimator_
res_xgb_tuned = evaluate(best_xgb, X_train, y_train, X_test, y_test, 'XGBoost (tuned)')

## 6. Model Comparison

In [ ]:
all_results = [res_rf_base, res_xgb_base, res_rf_tuned, res_xgb_tuned]
comp = pd.DataFrame([{
    'Model':       r['name'],
    'Test Acc':    round(r['acc'], 4),
    'CV Acc':      round(r['cv'], 4),
    'ROC-AUC':     round(r['roc_auc'], 4),
    'Train Time':  round(r['time'], 2)
} for r in all_results])
print(comp.to_string(index=False))

In [ ]:
# Grouped bar chart: Test Acc, CV Acc, ROC-AUC
metrics = ['Test Acc', 'CV Acc', 'ROC-AUC']
x = np.arange(len(comp))
width = 0.25
colors = sns.color_palette('muted', 3)

fig, ax = plt.subplots(figsize=(12, 5))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i * width, comp[metric], width, label=metric, color=color)
    for bar, val in zip(bars, comp[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(comp['Model'], rotation=15, ha='right')
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy & ROC-AUC')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Training time comparison
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(comp['Model'], comp['Train Time'], color=sns.color_palette('muted', len(comp)))
for bar, val in zip(bars, comp['Train Time']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}s', va='center', fontsize=9)
ax.set_xlabel('Training Time (s)')
ax.set_title('Training Time Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# Tuned models — side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, res in zip(axes, [res_rf_tuned, res_xgb_tuned]):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(res['name'])
plt.suptitle('Tuned Models — Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.show()

## 7. ROC Curves

In [ ]:
# One-vs-Rest ROC curves for tuned models
y_bin = label_binarize(y_test, classes=np.arange(n_classes))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
palette = sns.color_palette('tab10', n_classes)

for ax, res in zip(axes, [res_rf_tuned, res_xgb_tuned]):
    for i, cls in enumerate(le.classes_):
        fpr, tpr, _ = roc_curve(y_bin[:, i], res['y_prob'][:, i])
        roc_auc_i = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=palette[i], lw=2,
                label=f'{cls} (AUC={roc_auc_i:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves (OvR) — {res["name"]}')
    ax.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Learning Curves

In [ ]:
def plot_learning_curve(model, X, y, ax, title):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    tr_mean, tr_std = train_scores.mean(1), train_scores.std(1)
    va_mean, va_std = val_scores.mean(1),   val_scores.std(1)

    ax.plot(train_sizes, tr_mean, 'o-', label='Train', color='steelblue')
    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color='steelblue')
    ax.plot(train_sizes, va_mean, 'o-', label='Validation', color='coral')
    ax.fill_between(train_sizes, va_mean - va_std, va_mean + va_std, alpha=0.15, color='coral')
    ax.set_title(title)
    ax.set_xlabel('Training samples')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.set_ylim(0.7, 1.02)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_learning_curve(best_rf,  X, y, axes[0], 'Learning Curve — Random Forest (tuned)')
plot_learning_curve(best_xgb, X, y, axes[1], 'Learning Curve — XGBoost (tuned)')
plt.suptitle('Learning Curves', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette('muted', len(features))

for ax, res, model in zip(axes,
                           [res_rf_tuned, res_xgb_tuned],
                           [best_rf, best_xgb]):
    imp = pd.Series(model.feature_importances_, index=features).sort_values()
    imp.plot(kind='barh', ax=ax, color=palette)
    ax.set_title(f'Feature Importance — {res["name"]}')
    ax.set_xlabel('Importance')
    for i, v in enumerate(imp.values):
        ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Feature Importances Comparison', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: feature importance side-by-side
imp_df = pd.DataFrame({
    'Random Forest': best_rf.feature_importances_,
    'XGBoost':       best_xgb.feature_importances_
}, index=features)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(imp_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Feature Importance Heatmap')
plt.tight_layout()
plt.show()

## 10. Conclusions

- Both Random Forest and XGBoost achieve high accuracy on the Car Evaluation dataset, outperforming a single Decision Tree.
- **Random Forest** (bagging) builds independent trees in parallel and reduces variance; it is faster to train and less sensitive to hyperparameters.
- **XGBoost** (gradient boosting) builds trees sequentially, correcting residual errors; it typically achieves slightly higher accuracy at the cost of more tuning effort (`learning_rate`, `gamma`, regularisation).
- Hyperparameter tuning via `GridSearchCV` and `RandomizedSearchCV` consistently improves both models over their baselines.
- `safety` and `persons` are the most important features in both models, consistent with domain knowledge.
- Learning curves confirm that both tuned models generalise well with no significant overfitting on this dataset size.
- For production use, XGBoost is preferred when maximum accuracy is required; Random Forest is preferred when interpretability, speed, or robustness to hyperparameter choice matters.